# Model Evaluation and SHAP Interpretation
## Goal
Train a classification model, evaluate precision/recall/F1 and ROC-AUC, and interpret feature contributions with SHAP.

**Dataset:** Wisconsin Diagnostic Breast Cancer (bundled with scikit-learn; no download needed).  
**Target:** `malignant` vs `benign` encoded as target names supplied by scikit-learn.  
**Important:** This is an educational demonstration, not a clinical diagnostic tool.


In [ ]:
# If needed, install dependencies in your environment:
# %pip install -r ../requirements.txt

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
ASSET_DIR = "../assets"
os.makedirs(ASSET_DIR, exist_ok=True)

dataset = load_breast_cancer(as_frame=True)
X, y = dataset.data, dataset.target
class_names = dataset.target_names
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Target classes:", list(class_names))


In [ ]:
model = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=class_names, digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_score), 4))


In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, cmap="Blues", ax=ax
)
ax.set_title("Confusion Matrix")
fig.tight_layout()
fig.savefig(os.path.join(ASSET_DIR, "confusion_matrix.png"), dpi=160)
plt.show()


In [ ]:
# ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_score)
auc = roc_auc_score(y_test, y_score)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"Random Forest (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", label="Chance")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(os.path.join(ASSET_DIR, "roc_curve.png"), dpi=160)
plt.show()


In [ ]:
# Built-in Random Forest feature importance (a baseline, not SHAP)
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values().tail(15)
ax = importance.plot(kind="barh", figsize=(8, 6), title="Top 15 Random Forest Feature Importances")
ax.set_xlabel("Feature importance")
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "feature_importance.png"), dpi=160)
plt.show()


## SHAP explanations
SHAP may require a compatible version of NumPy/Numba. If installation fails, create a clean virtual environment and install the versions in `requirements.txt`. TreeExplainer is used because the model is a tree ensemble.


In [ ]:
import shap

explainer = shap.TreeExplainer(model)
# For binary classification, SHAP output shape can vary by SHAP version.
shap_values = explainer.shap_values(X_test)

# Normalize outputs across SHAP versions:
if isinstance(shap_values, list):
    # List entry for each class; explain class 1.
    sv = shap_values[1]
elif getattr(shap_values, "ndim", 0) == 3:
    # Common layout: samples x features x classes.
    sv = shap_values[:, :, 1]
else:
    sv = shap_values

# Global summary plot
shap.summary_plot(sv, X_test, show=False, max_display=15)
plt.tight_layout()
plt.savefig(os.path.join(ASSET_DIR, "shap_summary.png"), dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# Local explanation for one held-out sample
row_index = 0
print("Explained row index:", row_index)
print("True class:", class_names[int(y_test.iloc[row_index])])
print("Predicted class:", class_names[int(y_pred[row_index])])
print("Predicted probability for class 1:", round(float(y_score[row_index]), 4))

# Force plot: saved as HTML for interactive viewing in a browser.
base_values = explainer.expected_value
if isinstance(base_values, (list, np.ndarray)) and np.ndim(base_values) > 0:
    base_value = base_values[1]
else:
    base_value = base_values

force = shap.force_plot(base_value, np.asarray(sv)[row_index], X_test.iloc[row_index], matplotlib=False)
shap.save_html(os.path.join(ASSET_DIR, "shap_force_plot.html"), force)
print("Saved interactive local explanation to ../assets/shap_force_plot.html")


## Interpretation checklist
- **Precision:** Of samples predicted as a class, the fraction that truly belong to it.
- **Recall:** Of actual samples in a class, the fraction correctly found.
- **F1-score:** Harmonic mean of precision and recall.
- **Confusion matrix:** Counts correct and incorrect predictions by class.
- **ROC-AUC:** Measures ranking performance across classification thresholds; 1.0 is ideal and 0.5 is chance.
- **SHAP summary:** Each dot is a sample-feature contribution. Positive SHAP values push the model output toward the explained class; negative values push away from it. Color convention typically indicates low-to-high feature values.
- **SHAP force plot:** Explains one sample by showing contributions that move the model output away from the expected/base value.

Do not interpret these correlations as causal effects. In this dataset, features are measurements derived from digitized breast-mass images; confirm any domain claims against reliable medical literature. This notebook is for coursework and must not be used for patient care.
